In [ ]:
!pip install pyspark tqdm

In [ ]:
import requests
import time
import urllib.parse
import os
from tqdm import tqdm
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("WikipediaBigData") \
    .getOrCreate()

print("Spark Ready ✅")

Spark Ready ✅


In [ ]:
headers = {
    "User-Agent": "BigDataProject/1.0 (purikhai271@gmail.com)"
}

project = "en.wikipedia.org"
access = "all-access"
agent = "all-agents"
granularity = "daily"

start_date = "20150801"
end_date = "20260131"

output_file = "wiki_bigdata.csv"
progress_file = "progress.txt"

In [ ]:
import requests

headers = {
    "User-Agent": "MyProject (purikhai271@gmail.com)"
}

start_year = 2015
start_month = 8

end_year = 2026
end_month = 1

articles = set()  # biar ga duplikat

year = start_year
month = start_month

while (year < end_year) or (year == end_year and month <= end_month):

    month_str = str(month).zfill(2)

    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia/all-access/{year}/{month_str}/all-days"

    print(f"Ambil data: {year}-{month_str}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()

        for day in data["items"]:
            for art in day["articles"]:
                articles.add(art["article"])
    else:
        print(f"Gagal ambil {year}-{month_str}")

    # naik ke bulan berikutnya
    month += 1
    if month > 12:
        month = 1
        year += 1

print("Total artikel unik:", len(articles))

Ambil data: 2015-08
Ambil data: 2015-09
Ambil data: 2015-10
Ambil data: 2015-11
Ambil data: 2015-12
Ambil data: 2016-01
Ambil data: 2016-02
Ambil data: 2016-03
Ambil data: 2016-04
Ambil data: 2016-05
Ambil data: 2016-06
Ambil data: 2016-07
Ambil data: 2016-08
Ambil data: 2016-09
Ambil data: 2016-10
Ambil data: 2016-11
Ambil data: 2016-12
Ambil data: 2017-01
Ambil data: 2017-02
Ambil data: 2017-03
Ambil data: 2017-04
Ambil data: 2017-05
Ambil data: 2017-06
Ambil data: 2017-07
Ambil data: 2017-08
Ambil data: 2017-09
Ambil data: 2017-10
Ambil data: 2017-11
Ambil data: 2017-12
Ambil data: 2018-01
Ambil data: 2018-02
Ambil data: 2018-03
Ambil data: 2018-04
Ambil data: 2018-05
Ambil data: 2018-06
Ambil data: 2018-07
Ambil data: 2018-08
Ambil data: 2018-09
Ambil data: 2018-10
Ambil data: 2018-11
Ambil data: 2018-12
Ambil data: 2019-01
Ambil data: 2019-02
Ambil data: 2019-03
Ambil data: 2019-04
Ambil data: 2019-05
Ambil data: 2019-06
Ambil data: 2019-07
Ambil data: 2019-08
Ambil data: 2019-09


In [ ]:
try:
    with open(progress_file, "r") as f:
        start_index = int(f.read())
except:
    start_index = 0

print("Mulai dari index:", start_index)

Mulai dari index: 7522


In [ ]:
import os
import time
import requests
import urllib.parse
from tqdm import tqdm

headers = {
    "User-Agent": "MyProject (vinapuji2005@gmail.com)"
}

# ===== PARAMETER =====
project = "en.wikipedia"
access = "all-access"
agent = "user"
granularity = "daily"
start_date = "20150801"
end_date = "20260131"

output_file = "output.csv"
progress_file = "progress.txt"

# ===== FIX: pastikan articles bisa di-index =====
articles = list(articles)   # <-- INI PEMBETULAN UTAMA

# ===== LOAD PROGRESS =====
start_index = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as p:
        start_index = int(p.read().strip())

# ===== MODE FILE =====
mode = "a" if os.path.exists(output_file) else "w"

with open(output_file, mode, encoding="utf-8") as f:

    # header CSV
    if mode == "w":
        f.write("article,timestamp,views\n")

    for i in tqdm(range(start_index, len(articles))):

        article = articles[i]
        encoded = urllib.parse.quote(article, safe='')

        url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/{access}/{agent}/{encoded}/{granularity}/{start_date}/{end_date}"

        success = False

        for attempt in range(5):
            try:
                r = requests.get(url, headers=headers, timeout=15)

                if r.status_code == 200:
                    items = r.json().get("items", [])

                    for item in items:
                        f.write(f"{article},{item['timestamp']},{item['views']}\n")

                    f.flush()

                    # ===== FIX: progress harus i+1 =====
                    with open(progress_file, "w") as p:
                        p.write(str(i + 1))

                    success = True
                    break

                else:
                    print(f"Status {r.status_code} untuk {article}, retry...")
                    time.sleep(2)

            except requests.exceptions.RequestException:
                print("Internet putus, retry...")
                time.sleep(10)

        if not success:
            print(f"Gagal ambil data: {article}")

        time.sleep(0.5)  # delay biar aman dari rate limit

print("Selesai!")

100%|██████████| 18876/18876 [4:32:55<00:00,  1.15it/s]

Selesai!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
output_file = "/content/drive/MyDrive/output.csv"
progress_file = "/content/drive/MyDrive/progress.txt"